# UART FPGA Test Interface

Interface Python para testar comunicação UART com a Tang Nano / MAX II via porta serial.

## Configuração
- **Baud Rate:** 9600 bps
- **Data Bits:** 8
- **Parity:** Even (Par)
- **Stop Bits:** 1
- **Porta:** `/dev/ttyUSB0` (Linux) ou `COM3` (Windows)


## Implementação com PySerial

In [ ]:
import serial
import time

In [ ]:
class UARTInterface:
    def __init__(self, port="/dev/ttyUSB0", baudrate=9600, timeout=1):
        """Cria conexão serial com PySerial"""
        try:
            self.ser = serial.Serial(
                port=port,
                baudrate=baudrate,
                bytesize=serial.EIGHTBITS,
                parity=serial.PARITY_EVEN,      # Paridade par (FPGA usa)
                stopbits=serial.STOPBITS_ONE,
                timeout=timeout
            )
            print(f" Conectado em {port} @ {baudrate} bps (PySerial)")
            self.ser.reset_input_buffer()
            self.ser.reset_output_buffer()
        except Exception as e:
            print(f" Erro ao conectar: {e}")
            self.ser = None
    
    def send(self, data):
        """Envia dados"""
        if not self.ser:
            return False
        try:
            if isinstance(data, str):
                data = data.encode()
            self.ser.write(data)
            print(f" Enviado: {data} ({len(data)} bytes)")
            return True
        except Exception as e:
            print(f" Erro ao enviar: {e}")
            return False
    
    def receive(self, size=1):
        """Recebe dados"""
        if not self.ser:
            return None
        try:
            data = self.ser.read(size)
            if data:
                print(f" Recebido: {data} ({len(data)} bytes)")
            return data
        except Exception as e:
            print(f" Erro ao receber: {e}")
            return None
    
    def send_and_wait(self, data, wait_time=0.1):
        """Envia e aguarda resposta"""
        self.send(data)
        time.sleep(wait_time)
        return self.receive()
    
    def send_string(self, text):
        """Envia string"""
        return self.send(text.encode())
    
    def receive_string(self, size=1):
        """Recebe como string"""
        data = self.receive(size)
        return data.decode() if data else None
    
    def is_connected(self):
        """Verifica se está conectado"""
        return self.ser is not None and self.ser.is_open
    
    def close(self):
        """Fecha conexão"""
        if self.ser:
            self.ser.close()
            print(" Conexão fechada")

In [ ]:
print("TESTE COM PySerial")

try:
    # Teste Básico
    print("\n Teste Básico")
    uart = UARTInterface(port="/dev/ttyUSB0", baudrate=115200)
    
    if uart.is_connected():
        # Enviar um byte
        uart.send(b'\x41')  # ASCII 'A'
        time.sleep(0.2)
        
        # Receber
        response = uart.receive(1)
        
        uart.close()
    
    # Teste com String
    print("\n Teste com String")
    uart = UARTInterface(port="/dev/ttyUSB0", baudrate=115200)
    
    if uart.is_connected():
        uart.send_string("Hello FPGA!")
        time.sleep(0.5)
        response = uart.receive_string(20)
        uart.close()
    
    # Teste Múltiplos Bytes
    print("\n Teste Múltiplos Bytes")
    uart = UARTInterface(port="/dev/ttyUSB0", baudrate=115200)
    
    if uart.is_connected():
        test_data = bytes([0x01, 0x02, 0x03, 0x04, 0x05])
        uart.send(test_data)
        time.sleep(0.5)
        response = uart.receive(5)
        uart.close()
        
except Exception as e:
    print(f"Erro: {e}")

## Referência Rápida de Portas

- **Linux:** `/dev/ttyUSB0`, `/dev/ttyUSB1`, `/dev/ttyACM0`
- **Windows:** `COM3`, `COM4`, `COM5`
- **macOS:** `/dev/cu.usbserial-*`

### Como Encontrar a Porta?

**Linux:**
```bash
ls /dev/tty*
dmesg | grep -i usb
```

**Windows:**
```cmd
mode COM3
```

**macOS:**
```bash
ls -la /dev/cu.*
```

In [ ]:
"""
Teste Interativo - Use esta célula para testes manuais
"""

# Configurações
PORT = "/dev/ttyUSB0"
BAUDRATE = 115200

# Criar instância global
uart = UARTInterface(port=PORT, baudrate=BAUDRATE)

NameError: name 'UARTInterface' is not defined

In [ ]:
def test_loopback():
    """Testa se o FPGA está ecoando os dados (loopback)"""
    print("\n Testando Loopback...")
    test_messages = [
        b'A',
        b'Hello',
        bytes([0x01, 0x02, 0x03]),
        b'UART_TEST'
    ]
    
    for msg in test_messages:
        print(f"\nEnviando: {msg}")
        uart.send(msg)
        time.sleep(0.3)
        response = uart.receive(len(msg))
        
        if response == msg:
            print(" Loopback OK!")
        else:
            print(f" Esperado {msg}, recebido {response}")

In [ ]:
def test_baud_sync():
    """Testa sincronização de baud rate"""
    print("\n Testando Sincronização...")
    
    # Enviar sequência conhecida
    test_seq = b'\x55'  # 01010101 - padrão alternado
    uart.send(test_seq)
    time.sleep(0.5)
    response = uart.receive(1)
    
    if response:
        print(f" Dados recebidos com sucesso")
    else:
        print(" Timeout - verificar baud rate ou conexão")

In [ ]:
def manual_send(msg):
    """Envia mensagem manualmente"""
    if isinstance(msg, str):
        uart.send_string(msg)
    else:
        uart.send(msg)
    time.sleep(0.2)
    return uart.receive(len(msg) if isinstance(msg, bytes) else len(msg.encode()))

In [ ]:
# Exemplo: descomentar para usar
# test_loopback()
# test_baud_sync()
# response = manual_send("Test")

## Troubleshooting

### Problema: Porta não encontrada

```python
# Listar portas disponíveis
import serial.tools.list_ports
ports = serial.tools.list_ports.comports()
for port in ports:
    print(f"{port.device}: {port.description}")
```

### Problema: Timeout ao receber

- Verificar se FPGA está programado e rodando
- Verificar baud rate (deve ser 9600)
- Verificar parity (deve ser ODD)
- Testar com `minicom` ou `screen`

### Problema: Dados corrompidos

- Verificar cabos USB
- Aumentar `timeout`
- Verificar polaridade RX/TX

### Teste com Terminal

```bash
# Linux
minicom -D /dev/ttyUSB0 -b 9600
# ou
screen /dev/ttyUSB0 9600
```

---

## Estrutura UART do Projeto

Seu FPGA implementa:
- **TX:** Transmissão de dados (FSM com 4 estados)
- **RX:** Recepção de dados  
- **Parity:** Bit de paridade par (XOR de todos os bits)
- **Frame:** START + 8 bits + PARITY + STOP